# extracting feautres in csv format from audio files

In [6]:
import os
import librosa
import numpy as np
import pandas as pd
from tqdm import tqdm

def extract_audio_features(file_path):
    try:
        # Load audio file
        y, sr = librosa.load(file_path, mono=True, duration=30)
        
        # Extract features
        mfccs_mean = np.mean(librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13), axis=1)
        centroid_mean = np.mean(librosa.feature.spectral_centroid(y=y, sr=sr))
        zcr_mean = np.mean(librosa.feature.zero_crossing_rate(y))
        chroma_mean = np.mean(librosa.feature.chroma_stft(y=y, sr=sr))

        # Combine into one row
        feature_row = list(mfccs_mean) + [centroid_mean, zcr_mean, chroma_mean]
        return feature_row
    except Exception as e:
        # Silently skip unreadable files to keep the loop going
        return None

# --- UPDATED MAIN LOOP ---
# 1. Point directly to the FULL folder
dataset_path = 'VocalSet11/FULL' 

data = []
columns = [f'mfcc_{i}' for i in range(1, 14)] + ['spectral_centroid', 'zcr', 'chroma_mean', 'label']

# 2. Get all singer folders, ignoring weird files like .DS_Store
singer_folders = [d for d in os.listdir(dataset_path) if os.path.isdir(os.path.join(dataset_path, d))]

for singer in singer_folders:
    singer_dir = os.path.join(dataset_path, singer)
    print(f"Mining audio for singer: {singer}...")
    
    # 3. os.walk() searches ALL subfolders inside the singer's directory
    for root, dirs, files in os.walk(singer_dir):
        for file in files:
            # Check if it's a WAV file (ignoring case)
            if file.lower().endswith('.wav'):
                file_path = os.path.join(root, file)
                
                features = extract_audio_features(file_path)
                if features:
                    features.append(singer) # The label is the singer's name/ID
                    data.append(features)

# Save to CSV
df = pd.DataFrame(data, columns=columns)
df.to_csv('singer_features.csv', index=False)
print(f"SUCCESS! Created CSV with {len(data)} audio files extracted.")

Mining audio for singer: female1...
Mining audio for singer: female2...
Mining audio for singer: female3...
Mining audio for singer: female4...
Mining audio for singer: female5...
Mining audio for singer: female6...
Mining audio for singer: female7...
Mining audio for singer: female8...
Mining audio for singer: female9...
Mining audio for singer: male1...
Mining audio for singer: male10...
Mining audio for singer: male11...
Mining audio for singer: male2...
Mining audio for singer: male3...
Mining audio for singer: male4...
Mining audio for singer: male5...
Mining audio for singer: male6...
Mining audio for singer: male7...
Mining audio for singer: male8...
Mining audio for singer: male9...
SUCCESS! Created CSV with 3613 audio files extracted.


Checking directory: VocalSet11
Folder 'FULL' contains 21 files.
  -> Example file: .DS_Store
